# 08 — Quantization Export (FP32 → FP16 / INT8)

Xuất bản FP16 và INT8 (dynamic range quantization) cho 2 model đại diện đã chọn theo kết luận
notebook 05 (`yolo11n_640` — trade-off tốt nhất, `yolo11s_640` — accuracy cao nhất).

**Công cụ:** `ultralytics.YOLO.export(format="tflite", ...)`, pin `ultralytics==8.3.253` vì bản
`8.4.x` trở lên đã đổi export sang format `litert` và **bỏ hỗ trợ FP16** cho TFLite/LiteRT.

**Calibration set cho INT8:** dùng bộ 100 ảnh có sẵn trong
`AI/benchmark/pc_subset_dataset/` (không có bộ dataset gốc VR-TSD-2 đầy đủ trên máy này).

**Lưu ý môi trường Windows quan trọng:** pipeline export của ultralytics (onnx → onnx2tf →
SavedModel → TFLite) gọi các API ghi file cấp thấp của TensorFlow, và trên Windows các API này
có thể lỗi (`is not a directory`, `Failed to rename ...`) nếu đường dẫn làm việc:
- chứa ký tự Unicode có dấu (vd "Nhúng"), hoặc
- vượt quá giới hạn MAX_PATH (260 ký tự) khi cộng thêm các file tạm bên trong
  `<model>_saved_model/variables/variables_temp/...`.

Do đó notebook này **export vào một thư mục làm việc ngắn, thuần ASCII** (`C:\qexp` khi chạy),
rồi copy kết quả `.tflite` cuối cùng về `AI/models/deploy/`. Nếu chạy lại trên máy khác, chỉ cần
đổi `WORK_DIR` bên dưới sang một thư mục ngắn, không dấu, gần root ổ đĩa.


In [ ]:
import shutil
from pathlib import Path

from ultralytics import YOLO

REPO_ROOT = Path(r"D:\CITD\HK4\TTNT_Nhúng\Project\TrafficSign-Mobile-Benchmark")
TRAINED_DIR = REPO_ROOT / "AI" / "models" / "trained"
DEPLOY_DIR = REPO_ROOT / "AI" / "models" / "deploy"

# Thư mục làm việc ngắn, ASCII-only — xem lưu ý ở trên.
WORK_DIR = Path(r"C:\qexp")
CALIB_DATA_YAML = WORK_DIR / "calib" / "data.yaml"

MODELS = ["yolo11n_640", "yolo11s_640"]


In [ ]:
def export_one(model_name: str):
    pt_path = WORK_DIR / f"{model_name}_best.pt"
    print(f"
===== {model_name} =====")

    # ---- FP16 ----
    model = YOLO(str(pt_path))
    saved_dir = Path(model.export(format="tflite", imgsz=640, half=True))
    fp16_dst = DEPLOY_DIR / f"{model_name}_fp16.tflite"
    shutil.copy(saved_dir, fp16_dst)
    print("FP16 ->", fp16_dst, fp16_dst.stat().st_size / (1024 * 1024), "MiB")

    # ---- INT8 (dynamic range quantization) ----
    model = YOLO(str(pt_path))
    saved_dir2 = Path(
        model.export(format="tflite", imgsz=640, int8=True, data=str(CALIB_DATA_YAML))
    )
    int8_dst = DEPLOY_DIR / f"{model_name}_int8.tflite"
    shutil.copy(saved_dir2, int8_dst)
    print("INT8 ->", int8_dst, int8_dst.stat().st_size / (1024 * 1024), "MiB")


DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
for m in MODELS:
    export_one(m)
print("
Done.")


## Kết quả export

| Model | Kích thước (MiB) | Input layout |
|---|---|---|
| yolo11n_640 (FP32, baseline) | 10.178 | NCHW `[1,3,640,640]` |
| yolo11n_640_fp16 | 5.140 | NHWC `[1,640,640,3]` |
| yolo11n_640_int8 | 2.861 | NHWC `[1,640,640,3]` |
| yolo11s_640 (FP32, baseline) | 36.278 | NCHW `[1,3,640,640]` |
| yolo11s_640_fp16 | 18.191 | NHWC `[1,640,640,3]` |
| yolo11s_640_int8 | 9.488 | NHWC `[1,640,640,3]` |

**Quan trọng — khác biệt input layout:** 4 model FP32 gốc được export bằng pipeline khác
(ai-edge-torch, giữ nguyên NCHW như PyTorch). Pipeline `ultralytics.export()` dùng ở đây đi qua
ONNX → onnx2tf → TensorFlow nên theo layout chuẩn TensorFlow là NHWC. Cả hai layout đều hợp lệ với
TFLite; `LiteRTDetector.kt` phía Android đã được cập nhật để **tự nhận diện layout** theo tensor
shape của từng model (xem `inputLayout`/`InputLayout` trong file đó) nên không cần đồng bộ layout
giữa các bản.

**Về bản "INT8":** đây là **dynamic range quantization** (chỉ trọng số lượng tử hóa INT8, input/
output tensor vẫn FLOAT32) — đây là hành vi mặc định khi export INT8 qua `onnx2tf` với pipeline
này (onnx2tf tạo thêm bản full-integer nhưng ultralytics đổi tên bản dynamic-range thành `_int8`).
Do input/output vẫn FLOAT32, không cần thêm bước dequantize ở phía Android.
